# 第18章：接入 FastAPI 与 MySQL

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/04-%E6%8E%A5%E5%85%A5FastAPI%E4%B8%8EMySQL.md)整理，对应[视频 P29](https://www.bilibili.com/video/BV14rxM6qEnL?p=29)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 18.1 从离线校验到业务查询

调用链为Codex → MCP工具 → FastAPI → MySQL。业务API已负责身份、店铺和筛选，工具复用现有接口。核心工具：ecommerce_list_orders对应GET /orders，ecommerce_order_evidence对应单笔证据接口。

## 18.2 先直接调用HTTP接口

在已设置AGENT_TOKEN的PowerShell运行，API保持8000端口：

```powershell
$headers = @{ Authorization = "Bearer $env:AGENT_TOKEN" }
$queryUrl = "http://127.0.0.1:8000/orders?paid_on=2026-10-10&as_of=2026-10-11T12%3A00%3A00%2B08%3A00&offset=0&limit=20"
Invoke-RestMethod -Uri $queryUrl -Headers $headers | ConvertTo-Json -Depth 10
```

时间+号编码为%2B，避免被当空格。MCP适配使用HTTP参数编码，不需要模型手工拼URL。预期A1001/2/3，total表示符合超时条件的订单，不是当天全部订单。

## 18.3 查询参数与完整分页

paid_on为上海自然日，as_of必须有时区，offset>=0，limit在1–100。店铺不由任意参数指定，由Agent身份限定A。

读取本页以后沿next_offset继续到null。若中断，只报告已取部分并写未完成，不能宣称全量结果。

### 分页对报告完整性的影响

limit只控制本次取回条数，不缩小业务总数。total=3而items有2条时，仍有下一页；按next_offset请求，直到null才宣称清单已读完整。中途一页失败，则明确已取得多少、缺哪一段。

服务端固定店铺A，不暴露可任意填的store参数；日期与as_of由调用者明确，limit/offset受范围约束。这样模型能选择查询条件，又不能随意扩大身份边界。

## 18.4 让Agent真正读两页

要求查询固定日期和基准、每页2条、取全量、不建工单。首调用参数：

```json
{"paid_on":"2026-10-10","as_of":"2026-10-11T12:00:00+08:00","offset":0,"limit":2}
```

第一页items=A1001/A1002、total=3、next_offset=2；第二次offset=2返回A1003、next_offset=null。依据实际分页字段继续，不猜“应该没有更多”。

## 18.5 每单读取关联证据

```powershell
Invoke-RestMethod -Uri "http://127.0.0.1:8000/orders/A1001/evidence" -Headers $headers | ConvertTo-Json -Depth 10
```

MCP传 `{"order_id":"A1001"}`，返回order、inventory、fulfillment、analysis。对三单逐一取证：A1001需求2、现库存0，核实库存和补货；A1002库存20且warehouse_hold，核查拦截；A1003库存20、无履约记录，核查任务建立和同步，不直接定根因。

## 18.6 通过业务接口访问数据库

服务端固定字段和查询表达式统一业务口径、权限和稳定输出，不暴露任意SQL。表结构变更由数据层适配，工具契约兼容时Skill通常无需重写。

business.py的scope将上海付款日转成UTC左闭右开区间，并计算严格超时截止；predicate统一店铺、paid、日期和paid_at<deadline。序列化再转为+08:00；evidence依次判断无库存记录、库存不足、仓库拦截、证据不足。

### 从代码对应到业务判断

scope解析日期和带时区as_of，将上海自然日转换成UTC起止，并用as_of减24小时得到deadline；predicate以paid_at<deadline实现严格超过，而不是<=。list_orders同一条件分别算total和取分页列表，按订单id排序。

evidence先用store和order_id读取订单，再按store+sku取库存、order_id取履约。无库存记录与库存数为0不同：前者缺证据，后者是当前库存不足。返回事实对象与analysis分开，便于报告保持层次。

## 18.7 空查询与失败分别表达

| 结果 | 表达 |
| --- | --- |
| 成功且items为空 | 本次明确条件下无匹配订单 |
| 日期、时区、分页非法 | 参数校验失败，不能补造结果 |
| 无有效凭据 | 401认证失败 |
| B1001证据不可访问 | 404，不泄露其他店铺内容 |
| API连接失败或超时 | 未取得有效结果，不能写零异常 |

MCP将接口错误作为工具错误，不转换为空数组。写入超时还须处理执行结果不确定。

### HTTP状态码怎样影响后续动作

422说明参数结构或值需修正，不能继续输出虚构数据；401缺有效身份，403角色不符；404包括不存在和当前店铺不可见，不应根据错误探测其他店铺内容。409留给工作流状态或幂等冲突，后续需读具体detail。

网络错误没有成功业务结果，无法推出“没有异常”。当返回成功且items确为空时，才记录本次明确范围和时间下无匹配。保留这个区别，比报告语言流畅更重要。

## 18.8 实现文件与前端

以下保留business.py和api.py。API用Depends注入角色；Agent依赖返回店铺A，审批接口另检approver；Pydantic限制输入，异常映射409/404/422。生命周期加载配置、连接业务库、打开SqliteSaver并释放资源。

根路径返回static/index.html，/ui提供样式和脚本。完整工程已含这些资源，不需要npm构建；前端只调用接口，后端继续授权。工作流细节在第20章展开。

### API应用初始化与授权位置

lifespan在启动时加载配置、校验两令牌不同且非占位、创建检查点路径、建立业务引擎和SqliteSaver；退出时释放连接。前端和MCP使用相同业务应用，身份依赖决定允许访问的店铺或审批角色。

输入模型禁止额外字段，request_id还有长度和字符范围，时间必须带时区。这些是服务端可执行约束，Skill可以提醒但不能替代。根路径和/ui静态资源只是展示入口，实际查询和写入继续经过API。

## 18.9 business.py完整代码

固定过滤、分页与证据判断由API和工作流共同复用。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/business.py)。代码按资料快照保留。

```python
"""确定性的查询与证据分析，供 API 和工作流共同复用。"""
from datetime import date, datetime, timedelta, timezone
from sqlalchemy import select, and_, func
from app.db import orders, inventory, fulfillment

SHANGHAI = timezone(timedelta(hours=8))


def scope(paid_on: str, as_of: str):
    day = date.fromisoformat(paid_on)
    instant = datetime.fromisoformat(as_of)
    if instant.utcoffset() is None:
        raise ValueError("as_of 必须带时区")
    start = datetime.combine(day, datetime.min.time(), SHANGHAI)
    if instant < start:
        raise ValueError("as_of 不能早于所查支付日期起点")
    utc = lambda value: value.astimezone(timezone.utc).replace(tzinfo=None)
    return utc(start), utc(start + timedelta(days=1)), utc(instant - timedelta(hours=24))


def predicate(store, paid_on, as_of):
    start, end, deadline = scope(paid_on, as_of)
    return and_(orders.c.store == store, orders.c.status == "paid",
                orders.c.paid_at >= start, orders.c.paid_at < end,
                orders.c.paid_at < deadline)


def serialize_order(row):
    value = dict(row)
    value["paid_at"] = value["paid_at"].replace(tzinfo=timezone.utc).astimezone(SHANGHAI).isoformat()
    return value


def list_orders(engine, store, paid_on, as_of, offset=0, limit=20):
    condition = predicate(store, paid_on, as_of)
    with engine.connect() as conn:
        total = conn.scalar(select(func.count()).select_from(orders).where(condition))
        rows = conn.execute(select(orders).where(condition).order_by(orders.c.id)
                            .offset(offset).limit(limit)).mappings().all()
    next_offset = offset + len(rows) if offset + len(rows) < total else None
    return {"store": store, "paid_on": paid_on, "as_of": as_of,
            "total": total, "items": [serialize_order(row) for row in rows],
            "next_offset": next_offset}


def evidence(engine, store, order_id):
    with engine.connect() as conn:
        order = conn.execute(select(orders).where(orders.c.id == order_id,
                              orders.c.store == store)).mappings().first()
        if not order:
            raise LookupError("订单不存在或不在当前店铺权限内")
        stock = conn.execute(select(inventory).where(inventory.c.store == store,
                              inventory.c.sku == order["sku"])).mappings().first()
        event = conn.execute(select(fulfillment).where(fulfillment.c.order_id == order_id)).mappings().first()
    if stock is None:
        reason, suggestion = "库存记录缺失，原因待确认", "联系库存系统负责人补齐证据"
    elif stock["available"] < order["quantity"]:
        reason, suggestion = "当前可用库存不足，可能影响发货", "核实库存分配及补货计划"
    elif event and event["event"] == "warehouse_hold":
        reason, suggestion = "存在仓库拦截记录", "联系履约人员核验拦截原因"
    else:
        reason, suggestion = "现有证据不足，原因待确认", "检查履约同步和仓库处理状态"
    return {"order": serialize_order(order), "inventory": dict(stock) if stock else None,
            "fulfillment": dict(event) if event else None,
            "analysis": {"possible_reason": reason, "suggestion": suggestion}}
```

## 18.10 api.py完整代码

角色、参数、错误处理与检查点生命周期集中在业务服务。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/api.py)。代码按资料快照保留。

```python
"""FastAPI 是业务授权边界；仅示例身份，实际企业需替换为统一身份系统。"""
import os
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv
from fastapi import FastAPI, Depends, HTTPException, Query
from fastapi.responses import JSONResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, Field, ConfigDict, field_validator
from langgraph.checkpoint.sqlite import SqliteSaver
from sqlalchemy import select
from app import business
from app.db import make_engine, tickets, orders
from app.workflow import WorkflowService, Conflict


class StartInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    paid_on: str
    as_of: str
    request_id: str = Field(min_length=1, max_length=64, pattern=r"^[a-zA-Z0-9_-]+$")

    @field_validator("as_of")
    @classmethod
    def timezone_required(cls, value):
        from datetime import datetime
        if datetime.fromisoformat(value).utcoffset() is None:
            raise ValueError("as_of 必须带时区")
        return value


class DecisionInput(BaseModel):
    model_config = ConfigDict(extra="forbid")
    decision: Literal["approve", "reject"]


def create_app(database_url=None, checkpoint_path=None, agent_token=None, approver_token=None):
    @asynccontextmanager
    async def lifespan(app):
        load_dotenv()
        agent = agent_token or os.environ.get("AGENT_TOKEN")
        approver = approver_token or os.environ.get("APPROVER_TOKEN")
        if not agent or not approver or agent == approver or agent.startswith("replace-") or approver.startswith("replace-"):
            raise RuntimeError("请分别设置两个不同的非占位令牌")
        app.state.agent_token, app.state.approver_token = agent, approver
        path = Path(checkpoint_path or os.getenv("CHECKPOINT_PATH", "data/checkpoints.sqlite"))
        path.parent.mkdir(parents=True, exist_ok=True)
        engine = make_engine(database_url or os.environ["DATABASE_URL"])
        app.state.engine = engine
        try:
            with SqliteSaver.from_conn_string(str(path)) as saver:
                app.state.workflows = WorkflowService(engine, saver)
                yield
        finally:
            engine.dispose()

    app = FastAPI(title="电商订单异常排查 API", lifespan=lifespan)
    static_dir = Path(__file__).resolve().parent / "static"
    app.mount("/ui", StaticFiles(directory=static_dir), name="ui")

    @app.get("/", include_in_schema=False)
    def dashboard():
        return FileResponse(static_dir / "index.html")

    bearer = HTTPBearer(auto_error=False)

    def identity(credentials: HTTPAuthorizationCredentials | None = Depends(bearer)):
        import secrets
        if credentials and secrets.compare_digest(credentials.credentials, app.state.agent_token):
            return "agent"
        if credentials and secrets.compare_digest(credentials.credentials, app.state.approver_token):
            return "approver"
        raise HTTPException(401, "缺少有效身份凭据")

    def agent(role=Depends(identity)):
        if role != "agent":
            raise HTTPException(403, "此入口需要 Agent 身份")
        return "A"

    @app.exception_handler(Conflict)
    async def conflict_handler(request, error):
        return JSONResponse(status_code=409, content={"detail": str(error)})

    @app.exception_handler(LookupError)
    async def missing_handler(request, error):
        return JSONResponse(status_code=404, content={"detail": str(error)})

    @app.exception_handler(ValueError)
    async def value_handler(request, error):
        return JSONResponse(status_code=422, content={"detail": str(error)})

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.get("/orders")
    def get_orders(paid_on: str, as_of: str, offset: int = Query(0, ge=0),
                   limit: int = Query(20, ge=1, le=100), store=Depends(agent)):
        return business.list_orders(app.state.engine, store, paid_on, as_of, offset, limit)

    @app.get("/orders/{order_id}/evidence")
    def get_evidence(order_id: str, store=Depends(agent)):
        return business.evidence(app.state.engine, store, order_id)

    @app.post("/workflows")
    def start(body: StartInput, store=Depends(agent)):
        return app.state.workflows.start(store, **body.model_dump())

    @app.get("/workflows/{workflow_id}")
    def status(workflow_id: str, role=Depends(identity)):
        return app.state.workflows.status(workflow_id, "A")

    @app.post("/workflows/{workflow_id}/decision")
    def decide(workflow_id: str, body: DecisionInput, role=Depends(identity)):
        if role != "approver":
            raise HTTPException(403, "Agent 身份无审批权限")
        return app.state.workflows.decide(workflow_id, "A", body.decision)

    @app.post("/workflows/{workflow_id}/resume")
    def resume(workflow_id: str, store=Depends(agent)):
        return app.state.workflows.resume(workflow_id, store)

    @app.get("/tickets")
    def get_tickets(offset: int = Query(0, ge=0), limit: int = Query(20, ge=1, le=100), store=Depends(agent)):
        with app.state.engine.connect() as conn:
            rows = conn.execute(select(tickets).join(orders, tickets.c.order_id == orders.c.id)
                .where(orders.c.store == store).order_by(tickets.c.id).offset(offset).limit(limit)).mappings().all()
        return {"items": [dict(row) for row in rows], "offset": offset, "limit": limit}

    return app


app = create_app()
```